# 01. Data source and preparation

**Question: Are the available data suitable for a retrospective study of daily emergency department visits in the Rhône, and what are their limitations?**

The source is the [DREES corrected time series of emergency department visits, 2017–2023](https://www.data.gouv.fr/datasets/series-longues-corrigees-du-nombre-de-passages-aux-urgences-2017-a-2023-en-france/). 

In [30]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

dir = Path.cwd().parent
file = dir / "data" / "raw" / "emergency-visits-2017-2023.csv"

df_raw = pd.read_csv(file,sep=";")
df = df_raw[df_raw["dep"]=='69'].copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df = df.set_index("date")

# About the dates 

Our data is supposed to cover the period from January 1, 2017, to December 31, 2023, with one data point per day. In this code, we will verify that our DataFrame fully covers the defined range, with no dates outside that range and no duplicate data within it.


In [ ]:
expected_dates = pd.date_range("2017-01-01", "2023-12-31", freq="D")
observed_dates = pd.DatetimeIndex(df.index).normalize() 
#We're not supposed to have hours in our df["date"], but I prefer to normalize them to avoid any issues. 

In [41]:
missing = expected_dates.difference(observed_dates)
print(f"We have {len(missing)} missing dates")
out_of_range = observed_dates.difference(expected_dates)
print(f"We have {len(out_of_range)} dates out of rance")
duplicates = observed_dates[observed_dates.duplicated()]
print(f"We have {len(duplicates)} duplicates")


We have 0 missing dates
We have 0 dates out of rance
We have 0 duplicates


# About the visit counts

We will verify the quality of the visit count data. Visit counts must be numeric, non-missing, finite, and non-negative.

In [45]:
visits = pd.to_numeric(df["nb_passages"], errors="coerce")

raw_missing = int(df["nb_passages"].isna().sum())
print(f"{raw_missing} missing values")
non_numeric = int(visits.isna().sum()) - raw_missing
print(f"{non_numeric} non-numeric values")
non_finite = int((~np.isfinite(visits.dropna())).sum())
print(f"{non_finite} non-finite values")
negatives = int((visits < 0).sum())
print(f"{negatives} negative values")

0 missing values
0 non-numeric values
0 non-finite values
0 negative values


,dep,libelle_dep,nb_passages
date,,,
2017-01-01,69,Rhône,1733.8
2017-01-02,69,Rhône,1903.0
2017-01-03,69,Rhône,1763.9
2017-01-04,69,Rhône,1706.2
2017-01-05,69,Rhône,1751.3
...,...,...,...
2023-12-27,69,Rhône,1834.1
2023-12-28,69,Rhône,1824.1
2023-12-29,69,Rhône,1846.0
